In [8]:
import pandas as pd
import re
import json

def preprocess_gpt_oss_trace(trace: str) -> dict:
    """
    Process traces with GPT-OSS format tags to extract different sections.
    
    Args:
        trace: The trace string with channel tags
        
    Returns:
        Dictionary with extracted sections
    """
    result = {
        'analysis': '',
        'final_content': '',
        'full_trace': trace
    }

    if '<|channel|>analysis<|message|>' in trace:
        analysis_start = trace.find('<|channel|>analysis<|message|>') + len('<|channel|>analysis<|message|>')
        analysis_end = trace.find('<|end|>', analysis_start)
        if analysis_end != -1:
            result['analysis'] = trace[analysis_start:analysis_end].strip()
            
    if '<|channel|>final<|message|>' in trace:
        final_start = trace.find('<|channel|>final<|message|>') + len('<|channel|>final<|message|>')

        final_end = len(trace)
        if '<|return|>' in trace[final_start:]:
            final_end = final_start + trace[final_start:].find('<|return|>')
        result['final_content'] = trace[final_start:final_end].strip()
    
    return result

data = []
with open('results.jsonl', 'r') as f:
    for line in f:
        data.append(json.loads(line))


In [9]:
df_kernelbook = pd.read_csv('kernelbook.csv')
df_kernelbook['id'] = df_kernelbook.index.astype(str)

print(f"Kernelbook shape: {df_kernelbook.shape}")
print(f"Kernelbook columns: {df_kernelbook.columns.tolist()}")
print(f"First few IDs in kernelbook (from index): {df_kernelbook['id'].head().tolist()}")

processed_data = []

for item in data:
    # Extract basic fields
    row = {
        'id': item.get('id', ''),
        'qc': item.get('qc', ''),
        'error': item.get('error', '')
    }
    
    # Process the trace to extract sections
    trace = item.get('trace', '')
    processed_trace = preprocess_gpt_oss_trace(trace)

    # Rename final_content to reasoning
    row['reasoning'] = processed_trace['final_content']
    
    processed_data.append(row)

df_traces = pd.DataFrame(processed_data)

print(f"Loaded {len(df_traces)} reasoning traces")
print(f"Traces columns: {df_traces.columns.tolist()}")
print(f"\nFirst trace ID: {df_traces.iloc[0]['id']}")
print(f"First trace QC: {df_traces.iloc[0]['qc']}")
print(f"First reasoning length: {len(df_traces.iloc[0]['reasoning'])} chars")


Kernelbook shape: (18162, 4)
Kernelbook columns: ['python_code', 'triton_code', 'messages', 'id']
First few IDs in kernelbook (from index): ['0', '1', '2', '3', '4']
Loaded 1500 reasoning traces
Traces columns: ['id', 'qc', 'error', 'reasoning']

First trace ID: 3256
First trace QC: PASS
First reasoning length: 3384 chars


In [10]:
# Convert trace IDs to strings for matching
df_traces['id'] = df_traces['id'].astype(str)

# Merge the dataframes on 'id'
# Select only the columns we need from kernelbook: id, python_code, triton_code
df_combined = df_traces.merge(
    df_kernelbook[['id', 'python_code', 'triton_code']], 
    on='id', 
    how='inner'
)

print(f"\nCombined dataframe shape: {df_combined.shape}")
print(f"Combined columns: {df_combined.columns.tolist()}")
print(f"\nMatched {len(df_combined)} records out of {len(df_traces)} traces")

# Check if any traces didn't match
unmatched = len(df_traces) - len(df_combined)
if unmatched > 0:
    print(f"Warning: {unmatched} traces had no matching kernelbook entries")

# Reorder columns: python_code, triton_code, reasoning
df_final = df_combined[['id', 'python_code', 'triton_code', 'reasoning', 'qc', 'error']]

print(f"\nFinal dataframe columns: {df_final.columns.tolist()}")
print(f"\nFirst record preview:")
print(f"ID: {df_final.iloc[0]['id']}")
print(f"Python code length: {len(df_final.iloc[0]['python_code'])} chars")
print(f"Triton code length: {len(df_final.iloc[0]['triton_code'])} chars")
print(f"Reasoning length: {len(df_final.iloc[0]['reasoning'])} chars")
df_final.head()


Combined dataframe shape: (1500, 6)
Combined columns: ['id', 'qc', 'error', 'reasoning', 'python_code', 'triton_code']

Matched 1500 records out of 1500 traces

Final dataframe columns: ['id', 'python_code', 'triton_code', 'reasoning', 'qc', 'error']

First record preview:
ID: 3256
Python code length: 443 chars
Triton code length: 1574 chars
Reasoning length: 3384 chars


,id,python_code,triton_code,reasoning,qc,error
0,3256,import torch\nimport torch.jit\nimport torch.o...,import torch\nimport triton\nimport triton.lan...,**1. High‑level goal** \nMap the element‑wise...,PASS,None
1,13049,import math\nimport torch\nfrom torch import n...,import torch\nfrom torch._inductor.select_algo...,**1. High‑level goal** \nMap the `EqualConv2d...,PASS,None
2,2111,from torch.nn import Module\nimport math\nimpo...,import torch\nfrom torch._inductor.select_algo...,**1. High‑level goal** \nMap a 2‑D convolutio...,PASS,None
3,9207,import math\nimport torch\nimport numpy as np\...,import torch\nimport triton\nimport triton.lan...,**1. High‑level goal** \nMap the simple eleme...,PASS,None
4,13689,import torch\nfrom torch import nn\nimport tor...,import torch\nfrom torch._inductor.select_algo...,**1. High‑level goal** \nMap the sequential s...,PASS,None


In [11]:
import json

def create_messages(row):
    """Create messages list for each row"""
    messages = [
        {
            'role': 'system',
            'content': 'You are an expert in optimizing PyTorch code to Triton GPU kernels. Convert the provided PyTorch implementations to efficient Triton code.'
        },
        {
            'role': 'user',
            'content': f'Convert the following PyTorch code to Triton:\n\n```python\n{row["python_code"]}\n```'
        },
        {
            'role': 'assistant',
            'content': f'<think>{row["reasoning"]}</think>\n\n```python\n{row["triton_code"]}\n```'
        }
    ]
    return messages

df_final['messages'] = df_final.apply(create_messages, axis=1)

print("First row messages:")
print(json.dumps(df_final['messages'].iloc[0], indent=2))

First row messages:
[
  {
    "role": "system",
    "content": "You are an expert in optimizing PyTorch code to Triton GPU kernels. Convert the provided PyTorch implementations to efficient Triton code."
  },
  {
    "role": "user",
    "content": "Convert the following PyTorch code to Triton:\n\n```python\nimport torch\nimport torch.jit\nimport torch.onnx\nimport torch.nn\n\n\nclass SimpleClampModel(torch.nn.Module):\n\n    def __init__(self, min, max):\n        super(SimpleClampModel, self).__init__()\n        self.min = min\n        self.max = max\n\n    def forward(self, input):\n        return torch.clamp(input, self.min, self.max)\n\n\ndef get_inputs():\n    return [torch.rand([4, 4, 4, 4])]\n\n\ndef get_init_inputs():\n    return [[], {'min': 4, 'max': 4}]\n\n```"
  },
  {
    "role": "assistant",
    "content": "<think>**1. High\u2011level goal**  \nMap the element\u2011wise `torch.clamp(input, min, max)` on a 4\u2011D tensor (shape\u202f[4,4,4,4]) to a Triton kernel that reads

C:\Users\Aditya\AppData\Local\Temp\ipykernel_24996\398750116.py:21: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_final['messages'] = df_final.apply(create_messages, axis=1)


In [12]:
# Save the combined dataframe to CSV
output_file = 'processed_reasoning_traces.csv'
df_final.to_csv(output_file, index=False, encoding='utf-8')

print(f"Data saved to {output_file}")
print(f"Total records: {len(df_final)}")
print(f"\nColumn statistics:")
for col in df_final.columns:
    if col in ['python_code', 'triton_code', 'reasoning', 'messages']:
        avg_len = df_final[col].str.len().mean()
        print(f"  {col}: avg {avg_len:.0f} chars")
    else:
        print(f"  {col}: {df_final[col].nunique()} unique values")


Data saved to processed_reasoning_traces.csv
Total records: 1500

Column statistics:
  id: 1500 unique values
  python_code: avg 1386 chars
  triton_code: avg 5691 chars
  reasoning: avg 7058 chars
  qc: 1 unique values
  error: 0 unique values
  messages: avg 3 chars


# Load the data

In [13]:
import pandas as pd

df = pd.read_csv('processed_reasoning_traces.csv')
df.head()

,id,python_code,triton_code,reasoning,qc,error,messages
0,3256,import torch\nimport torch.jit\nimport torch.o...,import torch\nimport triton\nimport triton.lan...,**1. High‑level goal** \nMap the element‑wise...,PASS,NaN,"[{'role': 'system', 'content': 'You are an exp..."
1,13049,import math\nimport torch\nfrom torch import n...,import torch\nfrom torch._inductor.select_algo...,**1. High‑level goal** \nMap the `EqualConv2d...,PASS,NaN,"[{'role': 'system', 'content': 'You are an exp..."
2,2111,from torch.nn import Module\nimport math\nimpo...,import torch\nfrom torch._inductor.select_algo...,**1. High‑level goal** \nMap a 2‑D convolutio...,PASS,NaN,"[{'role': 'system', 'content': 'You are an exp..."
3,9207,import math\nimport torch\nimport numpy as np\...,import torch\nimport triton\nimport triton.lan...,**1. High‑level goal** \nMap the simple eleme...,PASS,NaN,"[{'role': 'system', 'content': 'You are an exp..."
4,13689,import torch\nfrom torch import nn\nimport tor...,import torch\nfrom torch._inductor.select_algo...,**1. High‑level goal** \nMap the sequential s...,PASS,NaN,"[{'role': 'system', 'content': 'You are an exp..."


In [14]:
print(df["messages"][0])

[{'role': 'system', 'content': 'You are an expert in optimizing PyTorch code to Triton GPU kernels. Convert the provided PyTorch implementations to efficient Triton code.'}, {'role': 'user', 'content': "Convert the following PyTorch code to Triton:\n\n```python\nimport torch\nimport torch.jit\nimport torch.onnx\nimport torch.nn\n\n\nclass SimpleClampModel(torch.nn.Module):\n\n    def __init__(self, min, max):\n        super(SimpleClampModel, self).__init__()\n        self.min = min\n        self.max = max\n\n    def forward(self, input):\n        return torch.clamp(input, self.min, self.max)\n\n\ndef get_inputs():\n    return [torch.rand([4, 4, 4, 4])]\n\n\ndef get_init_inputs():\n    return [[], {'min': 4, 'max': 4}]\n\n```"}, {'role': 'assistant', 'content': '<think>**1. High‑level goal**  \nMap the element‑wise `torch.clamp(input, min, max)` on a 4‑D tensor (shape\u202f[4,4,4,4]) to a Triton kernel that reads each element, applies the lower‑ and upper‑bound logic, and writes the res